# 01 — (Optional) Archive Nutrition5k to Google Drive

**You usually don't need to run this.** Notebook 03 stages its own copy of Nutrition5k to **local disk** from the public bucket, because the ~5k per-dish RGB-D folders can't be read off Drive's FUSE mount during extraction/training — the many-small-files workload aborts (`Errno 103`). A Drive→local bulk copy would hit the same wall, so 03 re-syncs from the bucket instead.

Run this notebook only if you want a **persistent raw archive** on Drive (offline inspection, or a hedge against the bucket going away). Note that doing so means the ~15–25 GB gets fetched twice — once here to Drive, once in notebook 03 to local disk.

Pulls only what the pipeline uses: **overhead RGB-D imagery, metadata, and split ids** (~15–25 GB). The full dataset is 181.4 GB; the difference is side-angle video we never read. License: CC BY 4.0.

Everything lands in the project Drive folder — **no Google Cloud account, project, or gcloud tooling involved**: the files come straight off the public bucket over plain HTTPS. The copy is **resumable** — rerun after any disconnect and it skips files already in Drive. No GPU needed for this notebook.

In [1]:
# Mount Drive and point DATA at the archive location. This notebook is the
# optional archival path: it writes the raw dataset to Drive. Training (notebook
# 03) stages its own local copy and does not read this — see the header.
from google.colab import drive
drive.mount('/content/drive')

# Set this to the mounted path of the project folder in YOUR Drive.
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

DATA = f'{DRIVE_ROOT}/data/nutrition5k'   # everything lands under here
!mkdir -p "{DATA}"
print('data root:', DATA)

Mounted at /content/drive
data root: /content/drive/MyDrive/physics-powered-portion-estimation/data/nutrition5k


In [2]:
# General idea: mirror the slice of the public Nutrition5k bucket the pipeline
# needs into DATA over plain HTTPS. Two helpers — list_objects (enumerate) and
# fetch (download one file, skipping any already present) — then a parallel
# driver. The bucket is public, so there's no gcloud, project, or auth.
import concurrent.futures
import pathlib
import requests

BUCKET = 'nutrition5k_dataset'
# Only what the pipeline reads — overhead RGB-D, mass/kcal metadata, split ids —
# not the ~160 GB of side-angle video we never touch.
PREFIXES = [
    'nutrition5k_dataset/imagery/realsense_overhead/',
    'nutrition5k_dataset/metadata/',
    'nutrition5k_dataset/dish_ids/',
]
API = f'https://storage.googleapis.com/storage/v1/b/{BUCKET}/o'


def list_objects(prefix):
    """Page through the bucket's JSON listing API, yielding (name, size) for
    every object under `prefix`. Unauthenticated GET — the bucket is public."""
    token = None
    while True:
        # maxResults caps the page size; nextPageToken walks to the next page.
        params = {'prefix': prefix, 'fields': 'items(name,size),nextPageToken', 'maxResults': 5000}
        if token:
            params['pageToken'] = token
        r = requests.get(API, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        for item in payload.get('items', []):
            yield item['name'], int(item.get('size', 0))
        token = payload.get('nextPageToken')
        if not token:   # no more pages — done
            break


def fetch(job):
    """Download one object unless an identical-size copy already exists — that
    size check IS the resumability, so a rerun skips completed files. Streams to
    a .part temp and atomically renames, so an interrupted transfer never leaves
    a truncated file that a later run would mistake for complete."""
    name, size = job
    dest = pathlib.Path(DATA) / name.removeprefix('nutrition5k_dataset/')
    if dest.exists() and dest.stat().st_size == size:
        return 0  # already synced — skip
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://storage.googleapis.com/{BUCKET}/{name}'
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20):   # 1 MiB chunks
                f.write(chunk)
        tmp.rename(dest)   # atomic swap — the file appears only once fully written
    return 1


# Enumerate everything up front (so the total is known before downloading), then
# fetch in parallel — 8 threads saturates the bucket without much memory.
jobs = [j for p in PREFIXES for j in list_objects(p)]
print(f'{len(jobs)} objects, {sum(s for _, s in jobs) / 1e9:.1f} GB total')

downloaded = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    for i, n in enumerate(pool.map(fetch, jobs), 1):
        downloaded += n
        if i % 500 == 0:
            print(f'{i}/{len(jobs)} checked, {downloaded} downloaded')
print(f'complete: {downloaded} new files (rest were already in Drive)')

10481 objects, 3.2 GB total
500/10481 checked, 0 downloaded
1000/10481 checked, 0 downloaded
1500/10481 checked, 0 downloaded
2000/10481 checked, 0 downloaded
2500/10481 checked, 0 downloaded
3000/10481 checked, 0 downloaded
3500/10481 checked, 0 downloaded
4000/10481 checked, 0 downloaded
4500/10481 checked, 0 downloaded
5000/10481 checked, 0 downloaded
5500/10481 checked, 0 downloaded
6000/10481 checked, 0 downloaded
6500/10481 checked, 0 downloaded
7000/10481 checked, 0 downloaded
7500/10481 checked, 0 downloaded
8000/10481 checked, 0 downloaded
8500/10481 checked, 0 downloaded
9000/10481 checked, 0 downloaded
9500/10481 checked, 0 downloaded
10000/10481 checked, 0 downloaded
complete: 0 new files (rest were already in Drive)


In [3]:
# Verify: ~5,000 dish folders, each with rgb.png + depth_raw.png, plus metadata CSVs.
!ls "{DATA}/imagery/realsense_overhead" | wc -l
!ls "{DATA}/imagery/realsense_overhead" | head -3
!ls "{DATA}/metadata"
!ls "{DATA}/dish_ids/splits" | head
!du -sh "{DATA}"

3490
dish_1556572657
dish_1556573514
dish_1556575014
dish_metadata_cafe1.csv  dish_metadata_cafe2.csv  ingredients_metadata.csv
depth_test_ids.txt
depth_train_ids.txt
rgb_test_ids.txt
rgb_train_ids.txt
3.0G	/content/drive/MyDrive/physics-powered-portion-estimation/data/nutrition5k
